In [2]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import PromptTemplate
import os
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnablePassthrough


In [11]:
openai_model1 = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=2048,
)


In [3]:
passthrough = RunnablePassthrough()

In [4]:
print(passthrough.invoke("Hello World!"))

Hello World!


In [5]:
print(passthrough.invoke(2 + 2))


4


In [6]:
prompt1 = PromptTemplate(
    template="Write  a joke about this topic: {topic}",
    input_variables=["topic"],
)


In [7]:
parser = StrOutputParser()

In [12]:
joke_gen_chain = RunnableSequence(prompt1, openai_model1, parser)

In [8]:
google_model2 = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    max_output_tokens=2048,
)


In [13]:
prompt2 = PromptTemplate(
    template="Explain the given joke, because I didn't get it : {joke}",
    input_variables=["joke"],
)


In [14]:
parallel_chain = RunnableParallel(
    {
        "joke": RunnablePassthrough(),
        "explanation": RunnableSequence(prompt2, google_model2, parser),
    }
)

In [15]:
combined_chain = RunnableSequence(joke_gen_chain, parallel_chain)

In [16]:
result = combined_chain.invoke({'topic' : 'AI'})

In [17]:
for key, value in result.items():
    print(f"{key} ----> {value}")


joke ----> Why did the AI go broke?

Because it couldn't find the right "byte" to invest in!
explanation ----> Here is why the joke is funny: it relies on a double meaning (a pun) with computer-related words.

1. **"Byte" vs. "Bite":** A **"byte"** is a fundamental unit of digital information in computers (spelled with a 'y'). However, it sounds exactly like the word **"bite"** (as in taking a bite of food, or figuratively, getting a "taste" of success).

2. **"Investing":** When people invest money, they are looking for a good opportunity or a "good bite" of the market. 

3. **The Punchline:** Because the AI is a computer, it deals in computer **bytes**. The joke plays on the idea that the AI was trying to make money by searching for computer data (*bytes*) instead of actual financial investments, so it lost all its money and went broke. 

In short: It's a nerdy wordplay on "byte" and "bite."
